In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
import random
from pathlib import Path

# Seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cuda


In [2]:
DATA_DIR = Path("/kaggle/input/ml-1m-csv")
MAX_SEQ_LEN, HIDDEN_UNITS = 50, 50
NUM_BLOCKS, NUM_HEADS, DROPOUT_RATE = 2, 1, 0.2
BATCH_SIZE, LEARNING_RATE, NUM_EPOCHS = 128, 0.001, 200
NUM_NEG_TEST = 100

In [3]:
# Load repo-equivalent CSV
ml1m_df = pd.read_csv(DATA_DIR / "ml-1m.csv")
data = ml1m_df.values.astype(int)
print(f"Loaded {len(data)} interactions from CSV")

num_users = int(np.max(data[:, 0]))
num_items = int(np.max(data[:, 1]))
print(f"Users: {num_users}, Items: {num_items}")

Loaded 562800 interactions from CSV
Users: 5180, Items: 3526


In [4]:
def build_sequences_and_split(data):
    user_seqs = defaultdict(list)
    for u, i in data:
        user_seqs[u].append(i)
    
    train_seqs, val_seqs, test_seqs = {}, {}, {}
    user_train_items = defaultdict(set)
    
    for uid, seq in user_seqs.items():
        if len(seq) < 3: continue
        train_seqs[uid] = seq[:-2]
        user_train_items[uid] = set(seq[:-2])
        val_seqs[uid] = [seq[-2]]
        test_seqs[uid] = [seq[-1]]
    
    print(f"Processed {len(train_seqs)} users")
    return train_seqs, val_seqs, test_seqs, user_train_items

train_seqs, val_seqs, test_seqs, user_train_items = build_sequences_and_split(data)

Processed 5180 users


In [5]:
class SASRecDataset(Dataset):
    def __init__(self, user_seqs, num_items, maxlen, user_train_items):
        self.user_seqs = user_seqs
        self.num_items = num_items
        self.maxlen = maxlen
        self.user_train_items = user_train_items
        self.users = list(user_seqs.keys())
    
    def __len__(self):
        return len(self.users)
    
    def __getitem__(self, idx):
        userid = self.users[idx]
        seq = self.user_seqs[userid]
        
        seq = seq[-self.maxlen:] if len(seq) > self.maxlen else seq
        seqlen = len(seq)
        padded_seq = [0] * (self.maxlen - seqlen) + seq
        
        input_seq = padded_seq
        pos_items = padded_seq[1:] + [0]
        
        user_items = self.user_train_items.get(userid, set())
        neg_seq = []
        for _ in range(self.maxlen):
            negitem = np.random.randint(1, self.num_items + 1)
            while negitem in user_items:
                negitem = np.random.randint(1, self.num_items + 1)
            neg_seq.append(negitem)
        
        return (torch.LongTensor(input_seq),
                torch.LongTensor(pos_items),
                torch.LongTensor(neg_seq))


In [9]:
class PointWiseFeedForward(nn.Module):
    """Position-wise Feed-Forward Network"""
    def __init__(self, hidden_units, dropout_rate):
        super().__init__()
        self.conv1 = nn.Conv1d(hidden_units, hidden_units, kernel_size=1)
        self.dropout1 = nn.Dropout(p=dropout_rate)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(hidden_units, hidden_units, kernel_size=1)
        self.dropout2 = nn.Dropout(p=dropout_rate)
    
    def forward(self, x):
        # (B,L,D) -> (B,D,L)
        x = x.transpose(-2, -1)
        x = self.dropout2(self.conv2(self.relu(self.dropout1(self.conv1(x)))))
        x = x.transpose(-2, -1)  # Back to (B,L,D)
        return x

class SASRec(nn.Module):
    """Self-Attentive Sequential Recommendation"""
    def __init__(self, num_items, hidden_units, num_blocks, num_heads, maxlen, dropout_rate):
        super().__init__()
        self.num_blocks = num_blocks  # FIXED!
        self.item_emb = nn.Embedding(num_items + 1, hidden_units, padding_idx=0)
        self.pos_emb = nn.Embedding(maxlen, hidden_units)
        self.emb_dropout = nn.Dropout(p=dropout_rate)
        
        # Transformer blocks
        self.attention_layers = nn.ModuleList([
            nn.MultiheadAttention(hidden_units, num_heads, dropout=dropout_rate, batch_first=True)
            for _ in range(num_blocks)
        ])
        self.forward_layers = nn.ModuleList([
            PointWiseFeedForward(hidden_units, dropout_rate) for _ in range(num_blocks)
        ])
        self.attention_layernorms = nn.ModuleList([
            nn.LayerNorm(hidden_units, eps=1e-8) for _ in range(num_blocks)
        ])
        self.forward_layernorms = nn.ModuleList([
            nn.LayerNorm(hidden_units, eps=1e-8) for _ in range(num_blocks)
        ])
        self.last_layernorm = nn.LayerNorm(hidden_units, eps=1e-8)
        
        self.init_weights()
    
    def init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
    
    def forward(self, input_seq):
        batch_size, seqlen = input_seq.size()
        
        # Positional encoding
        position_ids = torch.arange(seqlen, dtype=torch.long, device=input_seq.device)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, -1)
        
        # Embeddings
        seq_emb = self.item_emb(input_seq)  # (B,L,D)
        pos_emb = self.pos_emb(position_ids)
        seq_emb += pos_emb
        seq_emb = self.emb_dropout(seq_emb)
        
        # Causal mask
        attn_mask = torch.triu(torch.ones(seqlen, seqlen, device=input_seq.device), diagonal=1).bool()
        
        # Transformer blocks
        seq_output = seq_emb
        for i in range(self.num_blocks):
            # Self-attention + residual
            attn_lnorm = self.attention_layernorms[i](seq_output)
            attn_output, _ = self.attention_layers[i](attn_lnorm, attn_lnorm, attn_lnorm, attn_mask=attn_mask)
            seq_output = seq_output + attn_output
            
            # Feed-forward + residual
            ff_lnorm = self.forward_layernorms[i](seq_output)
            ff_output = self.forward_layers[i](ff_lnorm)
            seq_output = seq_output + ff_output
        
        seq_output = self.last_layernorm(seq_output)
        return seq_output
    
    def predict(self, input_seq, candidate_items):
        """Predict scores for candidate items (last position)"""
        seq_output = self.forward(input_seq)
        final_hidden = seq_output[:, -1, :]  # (B,D)
        candidate_emb = self.item_emb(candidate_items)  # (B,C,D)
        scores = torch.matmul(candidate_emb, final_hidden.unsqueeze(-1)).squeeze(-1)  # (B,C)
        return scores


In [10]:
train_dataset = SASRecDataset(train_seqs, num_items, MAX_SEQ_LEN, user_train_items)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

model = SASRec(num_items, HIDDEN_UNITS, NUM_BLOCKS, NUM_HEADS, MAX_SEQ_LEN, DROPOUT_RATE).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.98))
print(f"Params: {sum(p.numel() for p in model.parameters())}")


Params: 209950


In [11]:
def train_epoch(model, train_loader, optimizer, device):
    model.train()
    total_loss, num_batches = 0, 0
    
    for batch in train_loader:
        input_seq, pos_items, neg_items = [x.to(device) for x in batch]
        
        optimizer.zero_grad()
        seq_output = model(input_seq)
        logits = torch.matmul(seq_output, model.item_emb.weight.T)
        
        pos_logits = logits.gather(2, pos_items.unsqueeze(-1)).squeeze(-1)
        neg_logits = logits.gather(2, neg_items.unsqueeze(-1)).squeeze(-1)
        
        mask = (pos_items != 0).float()
        pos_loss = -torch.log(torch.sigmoid(pos_logits) + 1e-10) * mask
        neg_loss = -torch.log(1 - torch.sigmoid(neg_logits) + 1e-10) * mask
        loss = (pos_loss + neg_loss).sum(-1).mean()
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches

print("Training...")
for epoch in range(1, NUM_EPOCHS + 1):
    loss = train_epoch(model, train_loader, optimizer, device)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} Loss: {loss:.4f}")


Training...
Epoch  10 Loss: 31.6853
Epoch  20 Loss: 26.6856
Epoch  30 Loss: 23.7771
Epoch  40 Loss: 21.7201
Epoch  50 Loss: 20.3477
Epoch  60 Loss: 19.4476
Epoch  70 Loss: 18.8400
Epoch  80 Loss: 18.2463
Epoch  90 Loss: 17.8350
Epoch 100 Loss: 17.4619
Epoch 110 Loss: 17.2037
Epoch 120 Loss: 16.9363
Epoch 130 Loss: 16.8032
Epoch 140 Loss: 16.5823
Epoch 150 Loss: 16.3881
Epoch 160 Loss: 16.2389
Epoch 170 Loss: 16.0237
Epoch 180 Loss: 15.9213
Epoch 190 Loss: 15.8994
Epoch 200 Loss: 15.8519


In [12]:
def evaluate(model, train_seqs, test_seqs, user_train_items, num_items, maxlen, device, num_neg=100, k_list=[10, 20]):
    model.eval()
    hr_dict = {k: [] for k in k_list}
    ndcg_dict = {k: [] for k in k_list}
    
    with torch.no_grad():
        for userid in test_seqs:
            if userid not in train_seqs: continue
            
            seq = train_seqs[userid][-maxlen:] if len(train_seqs[userid]) > maxlen else train_seqs[userid]
            seqlen = len(seq)
            padded_seq = [0] * (maxlen - seqlen) + seq
            
            pos_item = test_seqs[userid][0]
            user_items = user_train_items.get(userid, set())
            neg_items = []
            while len(neg_items) < num_neg:
                negitem = np.random.randint(1, num_items + 1)
                if negitem not in user_items and negitem != pos_item:
                    neg_items.append(negitem)
            
            candidate_items = [pos_item] + neg_items
            input_seq = torch.LongTensor([padded_seq]).to(device)
            candidate_tensor = torch.LongTensor([candidate_items]).to(device)
            
            scores = model.predict(input_seq, candidate_tensor).squeeze(0)
            indices = torch.argsort(scores, descending=True).cpu().numpy()
            rank = np.where(indices == 0)[0][0] + 1
            
            for k in k_list:
                hr_dict[k].append(1.0 if rank <= k else 0.0)
                ndcg_dict[k].append(1.0 / np.log2(rank + 1) if rank <= k else 0.0)
    
    results = {f'HR@{k}': np.mean(hr_dict[k]) for k in k_list}
    results.update({f'NDCG@{k}': np.mean(ndcg_dict[k]) for k in k_list})
    return results


In [13]:
print("\nTest results:")
test_results = evaluate(model, train_seqs, test_seqs, user_train_items, num_items, 
                       MAX_SEQ_LEN, device, NUM_NEG_TEST)
print(f"HR@20:   {test_results['HR@20']:.4f}")
print(f"NDCG@10: {test_results['NDCG@10']:.4f}")
print(f"HR@10:   {test_results['HR@10']:.4f}")


Test results:
HR@20:   0.8485
NDCG@10: 0.4581
HR@10:   0.7232
